# Review Pulse — Exploratory Data Analysis

**Dataset:** UCI Sentiment Labelled Sentences  
**Sources:** Amazon, IMDB, Yelp  
**Goal:** Understand the data before training — class balance, text length distributions, vocabulary, and any quality issues.

---
## Table of Contents
1. Load & Merge Raw Data
2. Dataset Overview
3. Class Distribution
4. Review Length Analysis
5. Source-wise Comparison
6. Word Frequency & Vocabulary
7. Word Clouds
8. Data Quality Checks
9. Key Findings Summary

In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# ── Plot style ───────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

COLORS = {'Positive': '#1D9E75', 'Negative': '#D85A30', 'amazon': '#378ADD', 'imdb': '#7F77DD', 'yelp': '#D4537E'}

print('Libraries loaded.')

---
## 1. Load & Merge Raw Data

In [ ]:
# ── Paths — adjust RAW_DIR if running outside Docker ─────────
RAW_DIR = os.environ.get(
    'RAW_DIR',
    os.path.join('..', 'airflow', 'data', 'raw', 'sentiment labelled sentences')
)

files = {
    'amazon': os.path.join(RAW_DIR, 'amazon_cells_labelled.txt'),
    'imdb':   os.path.join(RAW_DIR, 'imdb_labelled.txt'),
    'yelp':   os.path.join(RAW_DIR, 'yelp_labelled.txt'),
}

frames = []
for source, path in files.items():
    df = pd.read_csv(path, sep='\t', header=None, names=['review', 'label'])
    df['source'] = source
    frames.append(df)
    print(f'{source}: {len(df)} rows loaded')

raw = pd.concat(frames, ignore_index=True)
raw['sentiment'] = raw['label'].map({0: 'Negative', 1: 'Positive'})

print(f'\nTotal rows: {len(raw)}')
raw.head()

---
## 2. Dataset Overview

In [ ]:
print('Shape:', raw.shape)
print('\nData types:')
print(raw.dtypes)
print('\nNull counts:')
print(raw.isnull().sum())
print('\nDuplicate rows:', raw.duplicated(subset=['review']).sum())
print('\nSample reviews:')
raw.sample(5, random_state=42)[['review', 'sentiment', 'source']]

---
## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Overall class balance ─────────────────────────────────────
counts = raw['sentiment'].value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=[COLORS[s] for s in counts.index],
                   width=0.5, edgecolor='none')
axes[0].set_title('Overall class distribution')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val} ({val/len(raw)*100:.1f}%)', ha='center', va='bottom', fontsize=11)

# ── Per-source class balance ──────────────────────────────────
source_sent = raw.groupby(['source', 'sentiment']).size().unstack(fill_value=0)
source_sent.plot(kind='bar', ax=axes[1],
                 color=[COLORS['Negative'], COLORS['Positive']],
                 width=0.6, edgecolor='none')
axes[1].set_title('Class distribution by source')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

print('\nClass balance (overall):')
print(raw['sentiment'].value_counts(normalize=True).round(3))

---
## 4. Review Length Analysis

Review length (in words) is the main feature used for drift detection. Understanding its distribution here validates the baseline stats computed by the Airflow pipeline.

In [ ]:
raw['word_count']  = raw['review'].apply(lambda x: len(str(x).split()))
raw['char_count']  = raw['review'].apply(lambda x: len(str(x)))

print('Word count statistics:')
print(raw['word_count'].describe().round(2))
print('\nChar count statistics:')
print(raw['char_count'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Overall word count distribution ──────────────────────────
axes[0].hist(raw['word_count'], bins=40, color='#378ADD', edgecolor='none', alpha=0.8)
axes[0].axvline(raw['word_count'].mean(),  color='#D85A30', linestyle='--', label=f"mean={raw['word_count'].mean():.1f}")
axes[0].axvline(raw['word_count'].median(),color='#1D9E75', linestyle='--', label=f"median={raw['word_count'].median():.1f}")
axes[0].set_title('Word count distribution (overall)')
axes[0].set_xlabel('Words per review')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=10)

# ── Word count by sentiment ───────────────────────────────────
for sent, grp in raw.groupby('sentiment'):
    axes[1].hist(grp['word_count'], bins=30, alpha=0.6,
                 color=COLORS[sent], label=sent, edgecolor='none')
axes[1].set_title('Word count by sentiment')
axes[1].set_xlabel('Words per review')
axes[1].legend()

# ── Word count by source (box plot) ──────────────────────────
source_order = ['amazon', 'imdb', 'yelp']
data_by_source = [raw[raw['source']==s]['word_count'].values for s in source_order]
bp = axes[2].boxplot(data_by_source, labels=source_order, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
for patch, src in zip(bp['boxes'], source_order):
    patch.set_facecolor(COLORS[src])
    patch.set_alpha(0.8)
axes[2].set_title('Word count by source')
axes[2].set_ylabel('Words per review')

plt.tight_layout()
plt.savefig('review_length_analysis.png', bbox_inches='tight')
plt.show()

---
## 5. Source-wise Comparison

In [ ]:
source_stats = raw.groupby('source').agg(
    total        =('review', 'count'),
    positive_pct =('label',  lambda x: round(x.mean()*100, 1)),
    avg_words    =('word_count', lambda x: round(x.mean(), 1)),
    median_words =('word_count', 'median'),
    avg_chars    =('char_count', lambda x: round(x.mean(), 1)),
    duplicates   =('review', lambda x: x.duplicated().sum()),
).reset_index()

print('Per-source summary:')
source_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Positive % per source ─────────────────────────────────────
axes[0].bar(source_stats['source'], source_stats['positive_pct'],
            color=[COLORS[s] for s in source_stats['source']],
            width=0.5, edgecolor='none')
axes[0].axhline(50, color='gray', linestyle='--', linewidth=1, label='50% line')
axes[0].set_title('Positive review % by source')
axes[0].set_ylabel('Positive %')
axes[0].set_ylim(0, 100)
for i, row in source_stats.iterrows():
    axes[0].text(i, row['positive_pct'] + 1.5, f"{row['positive_pct']}%",
                 ha='center', fontsize=11)

# ── Average word count per source ────────────────────────────
axes[1].bar(source_stats['source'], source_stats['avg_words'],
            color=[COLORS[s] for s in source_stats['source']],
            width=0.5, edgecolor='none')
axes[1].set_title('Average words per review by source')
axes[1].set_ylabel('Average word count')
for i, row in source_stats.iterrows():
    axes[1].text(i, row['avg_words'] + 0.3, str(row['avg_words']),
                 ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('source_comparison.png', bbox_inches='tight')
plt.show()

---
## 6. Word Frequency & Vocabulary

In [ ]:
# Basic cleaning for frequency analysis (mirrors the Airflow clean_data task)
def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'it','is','was','this','that','i','my','me','we','they','he','she',
    'not','be','are','have','has','had','do','did','so','as','if','by',
    'its','from','there','their','were','been','what','would','will',
    'you','your','just','all','more','very','can','about','up','out',
    'no','get','got','one','which','when','than','then','also','into',
    'other','like'
}

def get_tokens(df_subset):
    tokens = []
    for review in df_subset['review']:
        words = basic_clean(review).split()
        tokens.extend([w for w in words if w not in STOPWORDS and len(w) > 2])
    return tokens

pos_tokens = get_tokens(raw[raw['sentiment'] == 'Positive'])
neg_tokens = get_tokens(raw[raw['sentiment'] == 'Negative'])

print(f'Positive vocabulary size: {len(set(pos_tokens)):,}')
print(f'Negative vocabulary size: {len(set(neg_tokens)):,}')
print(f'Total unique tokens: {len(set(pos_tokens + neg_tokens)):,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, tokens, label, color in [
    (axes[0], pos_tokens, 'Positive', COLORS['Positive']),
    (axes[1], neg_tokens, 'Negative', COLORS['Negative']),
]:
    top20 = Counter(tokens).most_common(20)
    words, freqs = zip(*top20)
    ax.barh(range(len(words)), freqs, color=color, alpha=0.85, edgecolor='none')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_title(f'Top 20 words — {label} reviews')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('word_frequency.png', bbox_inches='tight')
plt.show()

---
## 7. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, tokens, label, color in [
    (axes[0], pos_tokens, 'Positive', COLORS['Positive']),
    (axes[1], neg_tokens, 'Negative', COLORS['Negative']),
]:
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap='Greens' if label == 'Positive' else 'Oranges',
        max_words=100,
        collocations=False,
    ).generate(' '.join(tokens))

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{label} reviews — word cloud', fontsize=14, pad=12)

plt.tight_layout()
plt.savefig('wordclouds.png', bbox_inches='tight')
plt.show()

---
## 8. Data Quality Checks

In [ ]:
print('=' * 50)
print('DATA QUALITY REPORT')
print('=' * 50)

# Nulls
nulls = raw.isnull().sum()
print(f'\nNull values:\n{nulls}')

# Duplicates
dupes = raw.duplicated(subset=['review']).sum()
print(f'\nDuplicate reviews: {dupes} ({dupes/len(raw)*100:.2f}%)')

# Very short reviews (< 3 words)
short = (raw['word_count'] < 3).sum()
print(f'Reviews with < 3 words: {short}')

# Very long reviews (> 100 words — potential outliers)
long = (raw['word_count'] > 100).sum()
print(f'Reviews with > 100 words: {long}')

# Invalid labels
invalid_labels = raw[~raw['label'].isin([0, 1])]
print(f'Rows with invalid labels: {len(invalid_labels)}')

# Cross-source duplicates
cross_dupes = raw[raw.duplicated(subset=['review'], keep=False)]
print(f'Reviews duplicated across sources: {len(cross_dupes)}')
if len(cross_dupes) > 0:
    print(cross_dupes[['review', 'source', 'sentiment']].head(5))

print('\nLabel distribution by source:')
print(raw.groupby(['source','sentiment']).size().unstack())

In [ ]:
# Show the shortest and longest reviews
print('5 shortest reviews:')
print(raw.nsmallest(5, 'word_count')[['review', 'word_count', 'sentiment', 'source']].to_string())

print('\n5 longest reviews:')
print(raw.nlargest(5, 'word_count')[['review', 'word_count', 'sentiment', 'source']].to_string())

---
## 9. Key Findings Summary

In [ ]:
pos_count = (raw['sentiment'] == 'Positive').sum()
neg_count = (raw['sentiment'] == 'Negative').sum()

print('=' * 55)
print('KEY FINDINGS')
print('=' * 55)
print(f'Total reviews       : {len(raw):,}')
print(f'Sources             : Amazon, IMDB, Yelp (500 each)')
print(f'Positive reviews    : {pos_count} ({pos_count/len(raw)*100:.1f}%)')
print(f'Negative reviews    : {neg_count} ({neg_count/len(raw)*100:.1f}%)')
print(f'Duplicate rows      : {raw.duplicated(subset=["review"]).sum()}')
print(f'Avg words/review    : {raw["word_count"].mean():.1f}')
print(f'Median words/review : {raw["word_count"].median():.0f}')
print(f'Vocabulary size     : {len(set(pos_tokens + neg_tokens)):,} unique tokens')
print('=' * 55)
print()
print('Observations:')
print('  - Dataset is nearly perfectly balanced (50/50 by design)')
print('  - IMDB reviews tend to be longer than Amazon/Yelp')
print('  - Positive reviews use words like: good, great, love, best, nice')
print('  - Negative reviews use words: bad, worst, terrible, waste, return')
print('  - No null values; some duplicate reviews exist across sources')
print()
print('Implications for modelling:')
print('  - Balanced classes => accuracy is a valid metric (no reweighting needed)')
print('  - TF-IDF with bigrams should capture discriminative phrases well')
print('  - Length variation across sources may cause mild distribution shift')
print('    when deploying on unseen data — drift detection threshold of 0.1 is appropriate')